####Lateral Join
Lateral join allows to query right dataframe for each row of the left dataframe.
Lateral joins are especially useful when:
1. You need per-parent Top-N child rows
2. You want to invoke TVFs with arguments derived from each row


1. Find the most recent booking for each member

```
+---------+----------+---------+---------------+-------------------+-----+
|member_id|first_name|last_name|  facility_name|         start_time|slots|
+---------+----------+---------+---------------+-------------------+-----+
```

In [0]:
from pyspark.sql.functions import col,expr

members_df = (
    spark.table("dev_catalog.spark_db.members")
    .filter("member_id >0")
    .select("member_id","first_name","last_name")
    .alias("m")
)

bookings_df = spark.table("dev_catalog.spark_db.bookings").alias("b")
facilities_df = spark.table("dev_catalog.spark_db.facilities").alias("f")

lat_member_booking_join_df = (
    members_df.lateralJoin(
        bookings_df.where("b.member_id== m.member_id")
        .orderBy(col("start_time").desc())
        .limit(1),None,'left')
    .select("m.member_id", "m.first_name", "m.last_name", "b.facility_id", "b.start_time", "b.slots")
).alias('mb')

result_df = (
    lat_member_booking_join_df.join(facilities_df, on=expr("mb.facility_id == f.facility_id"), how="left")
    .select("mb.member_id", "mb.first_name", "mb.last_name", "f.facility_name", "mb.start_time", "mb.slots")
)

result_df.display()

2. Find all students with more than 1 years of Spark knowledge from offline_var_students

In [0]:
students_df = spark.table("dev_catalog.spark_db.offline_var_students").alias("s")

result_df = (
    students_df.lateralJoin(spark.tvf.variant_explode("s.skills"))
)

result_df.display()